<a href="https://colab.research.google.com/github/goncalo-mateus/Credit-Risk-Scoring-Underwriting-Engine/blob/main/Distressed_Debt_Waterfall_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from typing import Dict, List
from dataclasses import dataclass
import ipywidgets as widgets
from IPython.display import display, clear_output

@dataclass
class Tranche:
    name: str
    seniority: int  # 1 is the most senior (e.g., Senior Secured)
    face_value: float
    claimed_amount: float

class DistressedWaterfall:
    """Core quantitative engine for liquidation waterfalls and secondary market ROI."""

    def __init__(self, total_recovery_value: float, capital_structure: List[Tranche]):
        self.total_recovery_value = total_recovery_value
        # Enforces Absolute Priority Rule (APR) by sorting tranches by seniority
        self.capital_structure = sorted(capital_structure, key=lambda t: t.seniority)

    def calculate_waterfall(self) -> List[Dict]:
        """Calculates liquidation proceeds distribution across capital structure tiers."""
        remaining_cash = self.total_recovery_value
        results = []

        for tranche in self.capital_structure:
            if remaining_cash <= 0:
                paid_out = 0.0
                recovery_rate = 0.0
            elif remaining_cash >= tranche.claimed_amount:
                paid_out = tranche.claimed_amount
                recovery_rate = 1.0
                remaining_cash -= paid_out
            else:
                paid_out = remaining_cash
                recovery_rate = paid_out / tranche.claimed_amount
                remaining_cash = 0.0

            results.append({
                "tranche": tranche.name,
                "seniority": tranche.seniority,
                "face_value": tranche.face_value,
                "claimed_amount": tranche.claimed_amount,
                "paid_out": paid_out,
                "unpaid": tranche.claimed_amount - paid_out,
                "recovery_rate": recovery_rate
            })

        return results

    def calculate_investor_returns(self, market_prices: Dict[str, float]) -> List[Dict]:
        """Calculates secondary market investor profit and ROI based on entry discount."""
        waterfall_results = self.calculate_waterfall()
        returns_analysis = []

        for row in waterfall_results:
            tranche_name = row["tranche"]
            paid_out = row["paid_out"]

            if tranche_name in market_prices:
                buy_price = market_prices[tranche_name]
                face_value = row["face_value"]

                total_cost = face_value * buy_price
                absolute_return = paid_out - total_cost
                roi = (absolute_return / total_cost) * 100 if total_cost > 0 else 0.0

                returns_analysis.append({
                    "tranche": tranche_name,
                    "purchase_cost": total_cost,
                    "recovery_received": paid_out,
                    "net_profit": absolute_return,
                    "roi_percentage": roi
                })

        return returns_analysis

# --- HUD INTERACTIVE WIDGETS (ENGLISH) ---
recovery_slider = widgets.FloatSlider(
    value=130.0, min=0.0, max=300.0, step=5.0,
    description='Recovery Assets ($):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='480px')
)

unsecured_price_slider = widgets.FloatSlider(
    value=0.40, min=0.0, max=1.0, step=0.05,
    description='Unsecured Buy Price ($):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='480px')
)

output_area = widgets.Output()

def update_dashboard(change):
    with output_area:
        clear_output(wait=True)

        total_rec = recovery_slider.value
        unsec_price = unsecured_price_slider.value

        capital_structure = [
            Tranche(name="Senior Secured Debt", seniority=1, face_value=100.0, claimed_amount=100.0),
            Tranche(name="Unsecured Bonds", seniority=2, face_value=50.0, claimed_amount=50.0),
            Tranche(name="Equity", seniority=3, face_value=20.0, claimed_amount=20.0)
        ]

        engine = DistressedWaterfall(total_recovery_value=total_rec, capital_structure=capital_structure)

        print("=== LIQUIDATION WATERFALL DISTRIBUTION REPORT ===")
        for row in engine.calculate_waterfall():
            print(f"• {row['tranche']:<20} | Paid: ${row['paid_out']:<6.1f} | Recovery Rate: {row['recovery_rate']*100:.1f}%")

        print("\n=== SECONDARY MARKET INVESTOR RETURNS ===")
        quotes = {"Unsecured Bonds": unsec_price}
        for inv in engine.calculate_investor_returns(quotes):
            print(f"• {inv['tranche']:<20} | Cost: ${inv['purchase_cost']:<6.1f} | Profit: ${inv['net_profit']:<6.1f} | ROI: {inv['roi_percentage']:.1f}%")

# Bind events and render HUD
recovery_slider.observe(update_dashboard, names='value')
unsecured_price_slider.observe(update_dashboard, names='value')

display(widgets.VBox([recovery_slider, unsecured_price_slider]), output_area)
update_dashboard(None)

Output()